<div align="center">

# Jackrong-llm-finetuning-guide 🌌
**An Educational, End-to-End LLM Fine-Tuning Pipeline for Beginners and Developers**

This notebook currently focuses on a fine-tuning guide for **Llama3.x- inside Kaggle notebooks.**

 **Llama-3.2-3B-Think-Zero: A Teaching Guide for Reasoning-Oriented GRPO**

🌐 **Language:**  🇬🇧 **English** 

🤗 **HuggingFace:** [Jackrong](https://huggingface.co/Jackrong)

<br>

[![Unsloth](https://img.shields.io/badge/Powered%20by-Unsloth-8A2BE2?style=flat-square)](https://github.com/unslothai/unsloth)
[![Google Colab](https://img.shields.io/badge/Environment-Google%20Colab-F9AB00?style=flat-square&logo=googlecolab&logoColor=white)](https://colab.research.google.com/)
[![PyTorch](https://img.shields.io/badge/Framework-PyTorch-EE4C2C?style=flat-square&logo=pytorch&logoColor=white)](https://pytorch.org/)
[![Hugging Face](https://img.shields.io/badge/Model%20Hub-Hugging%20Face-FFD21E?style=flat-square&logo=huggingface&logoColor=black)](https://huggingface.co/)
[![LoRA PEFT](https://img.shields.io/badge/Technique-LoRA%20%2F%20PEFT-007EC6?style=flat-square)](#)
[![Beginner Friendly](https://img.shields.io/badge/Level-Beginner%20Friendly-brightgreen?style=flat-square)](#)

</div>

---


## 🏗️ Technical Pipeline & Strategy

This project implements a state-of-the-art **Zero-to-Thinking** pipeline to instill deep reasoning capabilities into the `Llama-3.2-3B-Instruct` model, following the methodology of OpenAI o1 and DeepSeek-R1.

### 🚀 Training Workflow

1. **Stage 1: Cold-Start SFT (Supervised Fine-Tuning)**
   - **Objective**: Establishing a foundational behavioral policy for `<think>...</think><answer>\\boxed{...}</answer>` formatting.
   - **Dataset**: `OpenMathReasoning-mini` curated for strict reasoning chains.

2. **Stage 2: GRPO Reinforcement Learning**
   - **Mechanism**: Leveraging Group Relative Policy Optimization to optimize reasoning quality via a multi-faceted reward matrix.
   - **Reward Matrix**: Includes **Structure Shaping**, **Mathematical Accuracy** (using robust regex and distance-based scoring), and **Anti-Repetition** penalties.

3. **Stage 3: Automated Quantization & Export**
   - Seamless integration for LoRA merging and GGUF conversion (Q4_K_M & Q8_0) for efficient local inference.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")
import wandb 
wandb.login(key=secret_value_0)
print("Attempted to log in to W&B.")
import os
output_directory = "/kaggle/working/"
if not os.path.exists(output_directory):
    os.makedirs(output_directory)
print(f"Checkpoints will be saved to: {output_directory}")

In [ ]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
!pip install --upgrade -qqq uv
try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
except: get_numpy = "numpy"; get_pil = "pillow"
try: import subprocess; is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
except: is_t4 = False
get_vllm, get_triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.10.2", "triton")
!uv pip install -qqq --upgrade     unsloth {get_vllm} {get_numpy} {get_pil} torchvision bitsandbytes xformers
!uv pip install -qqq {get_triton}
!uv pip install "huggingface_hub>=0.34.0" "datasets==4.3.0
!uv pip install transformers==4.56.2
!uv pip install --no-deps trl==0.22.2

In [ ]:
import os
os.environ['UNSLOTH_VLLM_STANDBY'] = "1" # Unsloth Standby reduces VRAM by 30%+
from unsloth import FastLanguageModel
import torch
max_seq_length = 4096 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    load_in_fp8 = False, 
    gpu_memory_utilization=0.9,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

In [ ]:
# =========================================================
# ✅  Route B: Model autonomously outputs <think>...</think><answer>...</answer>
#    - Use system prompt + SFT cold-start + RL format reward to learn format
# =========================================================

THINK_OPEN  = "<think>"
THINK_CLOSE = "</think>"
ANSWER_OPEN  = "<answer>"
ANSWER_CLOSE = "</answer>"

system_prompt = f"""You are a helpful assistant.
You will be provided with a problem. Please follow this format exactly (do not add any extra text outside the tags):
{THINK_OPEN}
... step-by-step reasoning ...
{THINK_CLOSE}
{ANSWER_OPEN}\\boxed{{final numeric answer}}{ANSWER_CLOSE}
The <think> section can be short for easy problems, but must not be empty.
The final answer inside \\boxed{{}} should be a single number.
"""

def _escape_for_jinja_single_quote(s: str) -> str:
    return s.replace("\\", "\\\\").replace("'", "\\'").replace("\n", "\\n")

_SYSTEM_PROMPT_ESC = _escape_for_jinja_single_quote(system_prompt)

# ✅  Llama-3.x Chat Template
chat_template = (
    "{% if messages[0]['role'] == 'system' %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\n' + messages[0]['content'] + '<|eot_id|>' }}"
    "{% set loop_messages = messages[1:] %}"
    "{% else %}"
    "{{ '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n\\n' + '" + _SYSTEM_PROMPT_ESC + "' + '<|eot_id|>' }}"
    "{% set loop_messages = messages %}"
    "{% endif %}"

    "{% for message in loop_messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<|start_header_id|>user<|end_header_id|>\\n\\n' + message['content'] + '<|eot_id|>' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' + message['content'] + '<|eot_id|>' }}"
    "{% endif %}"
    "{% endfor %}"

    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\\n\\n' }}"
    "{% endif %}"
)

tokenizer.chat_template = chat_template
print("✅  Model must autonomously output <think>...</think><answer>...</answer>")


In [ ]:

tokenizer.apply_chat_template(
    [
        {"role" : "user", "content" : "What is 1+1?"},
        {"role" : "assistant", "content" : "<think>\nI think it's 2.\n</think>\n<answer>\\boxed{2}</answer>"},
        {"role" : "user", "content" : "What is 2+2?"},
    ],
    tokenize = False,
    add_generation_prompt = True,
)


In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
dataset = dataset.to_pandas()[
    ["expected_answer", "problem", "generated_solution"]
]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(dataset["expected_answer"]), errors = "coerce").notnull()
# Select only numbers
dataset = dataset.iloc[np.where(is_number)[0]]

dataset

In [ ]:
import re

# =========================================================
# ✅  Cold-start SFT Data Processing
#    Target output unified as:
#    <think> ... </think>
#    <answer>\boxed{...}</answer>
# =========================================================

def _extract_think_from_generated_solution(sol: str) -> str:
    """
    Extract 'reasoning part' from generated_solution of OpenMathReasoning-mini as much as possible.
    We only put expected_answer into <answer> to prevent solutions in generated_solution from polluting the format.
    """
    if sol is None:
        return ""
    s = str(sol)

    # 1) Prioritize extracting from <think>...</think> inside
    m = re.search(r"<think>(.*?)</think>", s, flags=re.DOTALL)
    if m is not None:
        s = m.group(1)

    # 2) Remove potential <answer>...</answer>
    s = re.sub(r"<answer>.*?</answer>", "", s, flags=re.DOTALL)

    # 3) Remove common 'final answer' fragments (leave mostly reasoning)
    s = re.sub(r"Final\s*Answer\s*:.*", "", s, flags=re.DOTALL | re.IGNORECASE)
    s = re.sub(r"####.*", "", s, flags=re.DOTALL)

    # 4) If \boxed, appears, truncate directly (prevent mixing answers into thinking)
    s = s.split(r"\boxed", 1)[0]

    return s.strip()


def format_dataset(x):
    expected_answer = str(x["expected_answer"]).strip()
    problem = x["problem"]

    thoughts = _extract_think_from_generated_solution(x["generated_solution"])

    # Prevent empty reasoning leading the model to learn <think>\n\n</think> (giving up behavior)
    if len(thoughts) < 5:
        thoughts = "I will solve the problem step by step."

    assistant = (
        f"{THINK_OPEN}\n{thoughts}\n{THINK_CLOSE}\n"
        f"{ANSWER_OPEN}\\boxed{{{expected_answer}}}{ANSWER_CLOSE}"
    )

    return [
        {"role" : "system",    "content" : system_prompt},
        {"role" : "user",      "content" : problem},
        {"role" : "assistant", "content" : assistant},
    ]


dataset["Messages"] = dataset.apply(format_dataset, axis = 1)
dataset["Messages"].iloc[0]


In [ ]:
tokenizer.apply_chat_template(dataset["Messages"][0], tokenize = False)

In [ ]:
dataset["N"] = dataset["Messages"].apply(lambda x: len(tokenizer.apply_chat_template(x)))

dataset = dataset.loc[dataset["N"] <= max_seq_length * 0.8].copy()
dataset.shape

In [ ]:
from datasets import Dataset

dataset["text"] = tokenizer.apply_chat_template(dataset["Messages"].values.tolist(), tokenize = False)
dataset = Dataset.from_pandas(dataset)
dataset

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_ratio = 0.04,
        #warmup_steps = 10,
        num_train_epochs = 2, # Set this for 1 full training run.
        #max_steps=0,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use this for WandB etc
    ),
)

In [ ]:
trainer.train()

In [ ]:
import gc
for _ in range(5):
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
text = tokenizer.apply_chat_template(
    dataset[0]["Messages"][:2],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 1,
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
)

In [ ]:
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 256,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

In [ ]:
del dataset
torch.cuda.empty_cache()
import gc
gc.collect()

## Transition: Cold-Start to GRPO Reinforcement Learning

The upper half of this notebook is the **cold-start stage**. We first use supervised fine-tuning to give the model a stable initial policy, so it learns the target reasoning format and does not start RL from a completely chaotic state. In practice, this stage teaches the model to produce responses in the structure `<think>...</think><answer>\\boxed{...}</answer>`, which makes the later reward design much easier to optimize.

The lower half now begins the **GRPO reinforcement learning stage**. Instead of only imitating training examples, the model will sample completions, receive rewards for strict formatting, mathematical correctness, and lower repetition, and then update its policy toward stronger reasoning behavior. This is the stage where the cold-start policy is refined into a more reliable reasoning model.


In [ ]:
from datasets import load_dataset
dataset = load_dataset("open-r1/DAPO-Math-17k-Processed", "en", split = "train")
dataset

In [ ]:
dataset[0]["prompt"]

In [ ]:
dataset[0]["solution"]

In [ ]:
import re

# =========================================================
# ✅  RL Dataset Answer Extraction (for accuracy reward)
# =========================================================

_NUM_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_SLASH_FRAC_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?/[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_LATEX_FRAC_RE = re.compile(r"[-+]?\\(?:d?frac)\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}")

def _extract_all_boxed(text: str):
    """
    Robust extraction of all \boxed{...}(supports nested braces)
    Returns list[str]
    """
    if text is None:
        return []
    s = str(text)
    out = []
    i = 0
    needle = r"\boxed{"
    while True:
        start = s.find(needle, i)
        if start == -1:
            break
        j = start + len(needle)
        depth = 1
        buf = []
        while j < len(s) and depth > 0:
            ch = s[j]
            if ch == "{":
                depth += 1
                buf.append(ch)
            elif ch == "}":
                depth -= 1
                if depth > 0:
                    buf.append(ch)
            else:
                buf.append(ch)
            j += 1
        if depth == 0:
            out.append("".join(buf).strip())
            i = j
        else:
            break
    return out

def extract_gt_answer(solution: str):
    """
    Normalize various solution formats into 'comparable answer strings' as much as possible.
    Priority: the last \boxed{...}
    Next: the last fraction/number
    """
    if solution is None:
        return None
    s = str(solution)

    boxes = _extract_all_boxed(s)
    if boxes:
        return boxes[-1]

    # Next: latex frac / a/b / the last number
    m = None
    for m in _LATEX_FRAC_RE.finditer(s):
        pass
    if m:
        return m.group(0)
    for m in _SLASH_FRAC_RE.finditer(s):
        pass
    if m:
        return m.group(0)

    nums = _NUM_RE.findall(s.replace(",", ""))
    if nums:
        return nums[-1]

    return s.strip()

# quick sanity check
extract_gt_answer(dataset[0]["solution"])


In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_gt_answer(x["solution"]),
})
dataset[0]


In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

In [ ]:
import re
import math

# =========================================================
# ✅  Reward (Route B): Strict Format + Numerical Accuracy + Weak Repetition Penalty
#    Target output (must strictly match):
#    <think> ... </think>
#    <answer>\boxed{...}</answer>
#    Must stop writing after </answer> (except for special tokens like eos/eot)
# =========================================================

THINK_OPEN  = "<think>"
THINK_CLOSE = "</think>"
ANSWER_OPEN  = "<answer>"
ANSWER_CLOSE = "</answer>"

# -------------------------
# 0) Debug: Sample Output Inspection
# -------------------------
def print_samples_reward(prompts, completions, **kwargs):
    completion_content = completions[0][0]["content"]

    print("\n" + "="*20 + " 🔍 [Monitor] Sample generated by model " + "="*20)
    print(f"❓ Prompt tail: ...{str(prompts[0])[-400:]}")
    print("-" * 10)
    print(f"🧠 Raw completion:\n{completion_content[:2048]}")
    print("="*60 + "\n")

    return [0.0] * len(completions)


# -------------------------
# 1) Universal Utils
# -------------------------
# Tokens allowed at the end that don't count as 'extra writing' (depends on if your generation backend decodes them)
ALLOWED_TRAILING_STRS = [
    getattr(tokenizer, "eos_token", None),
    getattr(tokenizer, "pad_token", None),
    "<|eot_id|>",
    "<|end_of_text|>",
]
ALLOWED_TRAILING_STRS = [x for x in ALLOWED_TRAILING_STRS if x]

def _strip_allowed_trailing(text: str) -> str:
    if text is None:
        return ""
    s = str(text).rstrip()
    changed = True
    while changed:
        changed = False
        for tok in ALLOWED_TRAILING_STRS:
            if s.endswith(tok):
                s = s[: -len(tok)].rstrip()
                changed = True
    return s

def _count(hay: str, needle: str) -> int:
    return 0 if hay is None else str(hay).count(needle)


# -------------------------
# 2) Numerical Parsing / Similarity
# -------------------------
_NUM_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_SLASH_FRAC_RE = re.compile(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?/[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?")
_LATEX_FRAC_RE = re.compile(r"[-+]?\\(?:d?frac)\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}\{[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?\}")

def _normalize(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().strip("$").strip()
    s = re.sub(r"\s+", "", s)
    return s

def _to_float(x: str):
    if x is None:
        return None
    s = _normalize(x).replace(",", "")
    if not s:
        return None

    # Pure numbers
    if re.fullmatch(r"[-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?", s):
        try:
            return float(s)
        except:
            return None

    # a/b
    m = re.fullmatch(r"([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)/([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)", s)
    if m:
        try:
            return float(m.group(1)) / float(m.group(2))
        except:
            pass

    # latex frac
    m = re.fullmatch(r"([-+]?)\\(?:d?frac)\{([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\}\{([-+]?\d+(?:\.\d+)?(?:[eE][-+]?\d+)?)\}", s)
    if m:
        try:
            sign = -1.0 if m.group(1) == "-" else 1.0
            return sign * float(m.group(2)) / float(m.group(3))
        except:
            pass

    # Fallback: take the last number
    nums = _NUM_RE.findall(s)
    if nums:
        try:
            return float(nums[-1])
        except:
            return None
    return None

def _similar(a: str, b: str) -> bool:
    af, bf = _to_float(a), _to_float(b)
    if af is not None and bf is not None:
        return math.isclose(af, bf, rel_tol=1e-6, abs_tol=1e-9)
    return _normalize(a) == _normalize(b)


# -------------------------
# 3) \boxed{...} Extraction
# -------------------------
def _extract_all_boxed(text: str):
    if text is None:
        return []
    s = str(text)
    out = []
    i = 0
    needle = r"\boxed{"
    while True:
        start = s.find(needle, i)
        if start == -1:
            break
        j = start + len(needle)
        depth = 1
        buf = []
        while j < len(s) and depth > 0:
            ch = s[j]
            if ch == "{":
                depth += 1
                buf.append(ch)
            elif ch == "}":
                depth -= 1
                if depth > 0:
                    buf.append(ch)
            else:
                buf.append(ch)
            j += 1
        if depth == 0:
            out.append("".join(buf).strip())
            i = j
        else:
            break
    return out

def _extract_last_boxed(text: str):
    boxes = _extract_all_boxed(text)
    return boxes[-1] if boxes else None


# -------------------------
# 4) Strict Format Parsing
# -------------------------
_STRICT_RE = re.compile(
    r"^\s*<think>\s*(?P<think>.*?)\s*</think>\s*<answer>\s*(?P<answer>.*?)\s*</answer>\s*$",
    flags=re.DOTALL,
)

_ONLY_BOXED_RE = re.compile(r"^\s*\\boxed\{.*\}\s*$", flags=re.DOTALL)

def _format_info(resp: str):
    raw = "" if resp is None else str(resp)
    stripped = _strip_allowed_trailing(raw)

    # Counts (used for 'extra tag heavy penalty')
    c_to = _count(stripped, THINK_OPEN)
    c_tc = _count(stripped, THINK_CLOSE)
    c_ao = _count(stripped, ANSWER_OPEN)
    c_ac = _count(stripped, ANSWER_CLOSE)

    # Sequence check (rough)
    p_to = stripped.find(THINK_OPEN)
    p_tc = stripped.find(THINK_CLOSE)
    p_ao = stripped.find(ANSWER_OPEN)
    p_ac = stripped.find(ANSWER_CLOSE)
    order_ok = (p_to != -1 and p_tc != -1 and p_ao != -1 and p_ac != -1 and p_to < p_tc < p_ao < p_ac)

    m = _STRICT_RE.match(stripped)

    think_text = m.group("think") if m else ""
    answer_text = m.group("answer") if m else ""

    boxed_in_answer = _extract_last_boxed(answer_text) if answer_text else None

    # answer only allows \boxed{...}(strict mode)
    only_boxed = bool(answer_text) and (_ONLY_BOXED_RE.match(answer_text) is not None) and (answer_text.count(r"\boxed{") == 1)

    # </answer> Tail after (strict match usually has no tail; non-strict is for penalty)
    tail_after_answer = ""
    if p_ac != -1:
        tail_after_answer = stripped[p_ac + len(ANSWER_CLOSE):].strip()

    # strict_ok: Exactly once + regex full match + correct order + answer contains only boxed + think is not empty
    strict_ok = (
        m is not None
        and order_ok
        and (c_to == c_tc == c_ao == c_ac == 1)
        and only_boxed
        and (len(think_text.strip()) >= 5)
        and (tail_after_answer == "")
    )

    return {
        "raw": raw,
        "stripped": stripped,
        "counts": (c_to, c_tc, c_ao, c_ac),
        "order_ok": order_ok,
        "strict_ok": strict_ok,
        "think_text": think_text,
        "answer_text": answer_text,
        "boxed_in_answer": boxed_in_answer,
        "only_boxed": only_boxed,
        "tail_after_answer": tail_after_answer,
    }


# =========================================================
# ✅  Reward A: Format Reward(Strict regex + shaping, avoid 'giving up')
# =========================================================
# Experience: If 'format error penalty' is too large, model tends to output nothing (as no output receives less penalty)
# So we set:
#   - Empty output is worst (more negative)
#   - Non-empty output, even with bad format, has a floor (still slightly better than empty)
FMT_EMPTY          = -2.0
FMT_NONEMPTY_FLOOR = -1.8

FMT_PARTIAL_CAP    =  0.6   # Max 0.6 for non-strict, prevent tag-spamming from overriding accuracy
FMT_STRICT_BONUS   =  1.0   # Strict match extra bonus

DUP_PENALTY_PER_KIND = 0.9  # Penalty once for each tag 'repetition' (heavy penalty but doesn't crash)
TAIL_PENALTY         = 0.4  # </answer> Extra content after

def basic_structure_reward(prompts, completions, **kwargs):
    scores = []
    for c in completions:
        r = c[0]["content"]
        info = _format_info(r)

        s = info["stripped"]
        if len(s.strip()) == 0:
            scores.append(FMT_EMPTY)
            continue

        c_to, c_tc, c_ao, c_ac = info["counts"]

        score = 0.05  # non-empty floor (encourage 'not giving up')

        # (1) partial shaping: occurrence/closure/sequence
        score += 0.15 if c_to >= 1 else -0.05
        score += 0.15 if c_tc >= 1 else -0.05
        score += 0.15 if c_ao >= 1 else -0.05
        score += 0.15 if c_ac >= 1 else -0.05
        score += 0.10 if info["order_ok"] else -0.10

        # (2) think non-empty
        score += 0.10 if len(info["think_text"].strip()) >= 5 else -0.05

        # (3) answer boxed inside answer (minor shaping)
        score += 0.10 if info["boxed_in_answer"] is not None else -0.05
        score += 0.05 if info["only_boxed"] else -0.05

        # (4) </answer> Must stop after
        if info["tail_after_answer"]:
            score -= TAIL_PENALTY
        else:
            score += 0.05

        # (5) Heavy penalty for extras (each tag > 1)
        dup_kinds = 0
        if c_to > 1: dup_kinds += 1
        if c_tc > 1: dup_kinds += 1
        if c_ao > 1: dup_kinds += 1
        if c_ac > 1: dup_kinds += 1
        if dup_kinds:
            score -= DUP_PENALTY_PER_KIND * dup_kinds

        # (6) Strict match bonus bonus
        if info["strict_ok"]:
            score += FMT_STRICT_BONUS
        else:
            # Non-strict cap
            if score > FMT_PARTIAL_CAP:
                score = FMT_PARTIAL_CAP

        # (7) floor: Any non-empty output shouldn't be worse than 'empty' (prevent giving up)
        if score < FMT_NONEMPTY_FLOOR:
            score = FMT_NONEMPTY_FLOOR

        scores.append(score)

    return scores


# =========================================================
# ✅  Reward B: Accuracy Reward(Prioritize parsing <answer> in \boxed)
# =========================================================
CORRECT_ANSWER   = 5.0
FORMAT_BONUS     = 1.0    # strict_ok Extra bonus when
CORRECT_NOBOXED  = 2.0    # Correct but no boxed (prevent too fragile at early stage)
WRONG_BASE       = -1.0
CLOSE_BONUS      = 1.5
CLOSE_K          = 3.0
WRONG_CAP        = -0.1

def _close_shaping(pred_str: str, gt_str: str) -> float:
    pf = _to_float(pred_str)
    gf = _to_float(gt_str)
    if pf is None or gf is None:
        return 0.0
    scale = max(1.0, abs(gf))
    rel_err = abs(pf - gf) / scale
    return CLOSE_BONUS * math.exp(-CLOSE_K * rel_err)

def _broadcast_answers(answer, n: int):
    if not isinstance(answer, (list, tuple)):
        return [answer] * n
    if len(answer) == n:
        return list(answer)
    if len(answer) > 0 and n % len(answer) == 0:
        k = n // len(answer)
        out = []
        for a in answer:
            out.extend([a] * k)
        return out
    return [answer[0]] * n

def check_answer_last_boxed(prompts, completions, answer, **kwargs):
    scores = []
    answer_list = _broadcast_answers(answer, len(completions))

    for c, true_ans in zip(completions, answer_list):
        r = c[0]["content"]
        info = _format_info(r)

        # 1) Highest priority: <answer> the last inside boxed
        pred = info["boxed_in_answer"]
        used_boxed = pred is not None

        # 2) If no boxed, but answer_text exists: try to extract numbers directly
        if pred is None and info["answer_text"]:
            cand = None
            m = None
            for m in _LATEX_FRAC_RE.finditer(info["answer_text"]):
                pass
            if m: cand = m.group(0)
            if cand is None:
                for m in _SLASH_FRAC_RE.finditer(info["answer_text"]):
                    pass
                if m: cand = m.group(0)
            if cand is None:
                nums = _NUM_RE.findall(info["answer_text"].replace(",", ""))
                if nums: cand = nums[-1]
            pred = cand
            used_boxed = False

        # 3) Fallback: last in the entire text boxed / the last number
        if pred is None:
            pred = _extract_last_boxed(info["stripped"])
            used_boxed = pred is not None
        if pred is None:
            nums = _NUM_RE.findall(info["stripped"].replace(",", ""))
            pred = nums[-1] if nums else None
            used_boxed = False

        if pred is None or true_ans is None:
            scores.append(WRONG_BASE)
            continue

        if _similar(pred, true_ans):
            s = CORRECT_ANSWER if used_boxed else CORRECT_NOBOXED
            if info["strict_ok"]:
                s += FORMAT_BONUS
            scores.append(s)
        else:
            val = _close_shaping(pred, true_ans)
            wrong = WRONG_BASE + val
            if wrong > WRONG_CAP:
                wrong = WRONG_CAP
            scores.append(wrong)

    return scores


# =========================================================
# ✅  Reward C: Weak Repetition Penalty
# =========================================================
def _simple_tokens(text: str):
    if text is None:
        return []
    return re.findall(r"\\[A-Za-z]+|[A-Za-z]+|\d+|[\u4e00-\u9fff]+", str(text))

def _repetition_rate(tokens, n: int) -> float:
    if len(tokens) < n + 12:
        return 0.0
    ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    if not ngrams:
        return 0.0
    return (len(ngrams) - len(set(ngrams))) / float(len(ngrams))

def ngram_repetition_penalty(prompts, completions, **kwargs):
    scores = []
    COEF_3, COEF_4, CAP = 0.005, 0.008, -0.15
    for c in completions:
        r = c[0]["content"]
        toks = _simple_tokens(r)
        penalty = -(COEF_3 * _repetition_rate(toks, 3) + COEF_4 * _repetition_rate(toks, 4))
        if penalty < CAP:
            penalty = CAP
        scores.append(penalty)
    return scores

print("✅  Reward functions updated for Route B (anti-silence + strict format).")


In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    #seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    #vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    #max_grad_norm = 0.1,
    #beta = 0.04,
    lr_scheduler_type = "cosine",
    optim = "paged_adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 6,
    gradient_accumulation_steps = 6, # Increase to 4 for smoother training
    num_generations = 6, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 1500,
    save_steps = 20,
    save_total_limit = 1,
    save_strategy = "steps",
    report_to = "wandb", # Can use Weights & Biases
    output_dir = output_directory,

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

In [ ]:
from trl import GRPOTrainer

# 1. Define Trainer (Your provided part)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        print_samples_reward,
        basic_structure_reward,        # Reward 1: Structural shaping (light, stable)
        check_answer_last_boxed,       # Reward 2: Main signal (boxed numerical check + weak fallback)
        ngram_repetition_penalty,      # Reward 4: Weak Repetition Penalty
        # think_boxed_alignment_reward, # Reward 3: Placeholder (can be omitted for now)
    ],
    args=training_args,
    train_dataset=dataset,
)

trainer.train()
print(train_result)

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.3,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

In [ ]:
text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Calculate pi."},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

In [ ]:
# ==========================================
# ✅  3. Save LoRA Adapter After Training
# ==========================================
print("💾 Saving GRPO after LoRA...")
trainer.model.save_lora("grpo_saved_lora_final")
print("✅  Savingcompleted!")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import whoami

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "HF_TOKEN was not found in Kaggle Secrets. Add a secret named 'HF_TOKEN'."
    ) from e

try:
    user_info = whoami(token=hf_token)
    username = user_info["name"]
    repo_id = f"{username}/Llama-3.2-3B-R1-Zero"
except Exception as e:
    raise RuntimeError(
        "Hugging Face authentication failed. Check HF_TOKEN or network access."
    ) from e

print(f"Using Hugging Face account: {username}")
print(f"Uploading merged 16-bit model to: {repo_id}")

model.push_to_hub_merged(
    repo_id,
    tokenizer,
    save_method="merged_16bit",
    token=hf_token,
)

print(f"Upload finished: https://huggingface.co/{repo_id}")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, whoami

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "HF_TOKEN was not found in Kaggle Secrets. Add a secret named 'HF_TOKEN'."
    ) from e

try:
    user_info = whoami(token=hf_token)
    username = user_info["name"]
    repo_id = f"{username}/Llama-3.2-3B-R1-Zero-GGUF"
except Exception as e:
    raise RuntimeError(
        "Hugging Face authentication failed. Check HF_TOKEN or network access."
    ) from e

print(f"Using Hugging Face account: {username}")
print(f"Uploading GGUF files to: {repo_id}")

api = HfApi()
api.create_repo(repo_id=repo_id, token=hf_token, exist_ok=True)

def find_gguf_files(workdir: str, quant_method: str):
    matches = []
    for name in os.listdir(workdir):
        if name.endswith(".gguf") and quant_method.lower() in name.lower():
            matches.append(os.path.join(workdir, name))
    return sorted(matches)

def convert_and_upload_gguf(quant_method: str):
    workdir = os.getcwd()
    print(f"Checking GGUF files in: {workdir}")
    gguf_files = find_gguf_files(workdir, quant_method)

    if not gguf_files:
        print(f"No existing {quant_method} GGUF file found. Starting conversion.")
        model.save_pretrained_gguf(
            ".",
            tokenizer,
            quantization_method=quant_method,
        )
        gguf_files = find_gguf_files(workdir, quant_method)

    if not gguf_files:
        raise RuntimeError(
            f"No GGUF file containing '{quant_method}' was found after conversion."
        )

    print(f"Found {len(gguf_files)} GGUF file(s) for {quant_method}.")

    for local_path in gguf_files:
        repo_path = os.path.basename(local_path)
        print(f"Uploading {repo_path} to {repo_id}")
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=repo_id,
            token=hf_token,
        )

    print(f"Finished uploading {quant_method} GGUF files.")

print("Starting GGUF export and upload.")

convert_and_upload_gguf("q4_k_m")
# Skip q8_0 if Kaggle storage is too tight.
#convert_and_upload_gguf("q8_0")

print(f"GGUF upload finished: https://huggingface.co/{repo_id}")